In [ ]:
import logging

FORMAT = '%(asctime)s %(name)s %(funcName)s %(message)s'
log_level = logging.WARNING
logging.basicConfig(format=FORMAT, datefmt='%H:%M:%S',
                    level=log_level)

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import h5py
import numpy as np
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import csv
from pathlib import Path 

bnsi_path = '/scicore/home/nimwegen/degroo0000/Bonsai-data-representation'
sys.path.append(bnsi_path)
from bonsai_scout.bonsai_scout_helpers import Bonvis_figure, Bonvis_settings, Bonvis_metadata

In [ ]:
from paper_figure_scripts_and_notebooks.simulating_datasets.analyzing_simulated_datasets.knn_recall_helpers import get_pdists_on_tree, Dataset, \
    compare_nearest_neighbours_to_truth, compare_pdists_to_truth, get_pdists_on_tree, compare_pdists_to_truth_per_cell, compare_nearest_neighbours_to_truth
from bonsai.bonsai_helpers import find_latest_tree_folder
from scipy.spatial import distance
from bonsai.bonsai_dataprocessing import SCData, get_bonsai_euclidean_distances
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib import colormaps
plt.set_loglevel(level='warning')

#### List information on datasets

In [ ]:
dataset_ids = [
    'simulated_binary_12_gens_samplingnoise_seed_2462',
    'simulated_binary_13_gens_samplingnoise_seed_2462',
    'simulated_binary_14_gens_samplingnoise_seed_2462',
    'simulated_binary_15_gens_samplingnoise_seed_2462',
]
input_data_folders = [os.path.join('/scicore/home/nimwegen/degroo0000/sc_datasets/simulated_datasets_timescaling', dataset_id) for dataset_id in dataset_ids]
bonsai_results_folders = [os.path.join('/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/slurm_runs_pipeline/output/simulated_datasets_timescaling', dataset_id, 'bonsai') 
                          for dataset_id in dataset_ids]
bonsai_results_folders_iterative = [os.path.join('/scicore/home/nimwegen/degroo0000/Bonsai-data-representation/slurm_runs_pipeline/output/simulated_datasets_timescaling_iterative', dataset_id, 'bonsai') 
                          for dataset_id in dataset_ids]

SINGLE_DATASET = False
dataset_descr_lst = ['4096 cells', '8192 cells', '16384 cells', '32768 cells']

## Create Bonsai visualization of ground truth dataset

In [ ]:
dataset_ids_gt = dataset_ids
dataset_descr_lst_gt = dataset_descr_lst
input_data_folders_gt = input_data_folders
bonsai_results_folders_gt = [os.path.join('/scicore/home/nimwegen/degroo0000/sc_datasets/simulated_datasets_timescaling', ds_id, 'true_tree') for ds_id in dataset_ids_gt]

## Get for each dataset a subset of cells for which we plot distances

In [ ]:
def read_ids(filepath):
    if not os.path.exists(filepath):
        print("Can't find IDs stored at {}.".format(filepath))
        return []
    ids = []
    with open(filepath, 'r') as file:
        reader = csv.reader(file, delimiter="\t")
        for row in reader:
            ids.append(row[0])
    return ids

In [ ]:
import random
N_SUBSET = 1000
cell_ids_selected_lst = []
cell_ids_gt_lst = []
cell_inds_selected_gt_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders):
    cell_ids = read_ids(os.path.join(input_data_folder, 'cellID.txt'))
    cell_ids_gt_lst.append(cell_ids)
    cell_ids_selected = random.sample(cell_ids, N_SUBSET)
    cell_ids_selected_lst.append(cell_ids_selected)
    cellid2ind = {cid: cind for cind, cid in enumerate(cell_ids)}
    cell_inds_selected = np.array([cellid2ind[cid] for cid in cell_ids_selected])
    cell_inds_selected_gt_lst.append(cell_inds_selected)

## Create pairwise distance plots

In [ ]:
# Read in ground truth squared pairwise distances (divided by the number of dimensions)
true_dists_lst = []
cell_ids_gt_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders):
    print(ind_dataset)
    cell_inds_selected_gt = cell_inds_selected_gt_lst[ind_dataset]
    npy_name = 'delta_true_{}.npy'.format(dataset_ids[ind_dataset])
    if os.path.exists(npy_name):
        delta_gc_true = np.load(npy_name)
    else:
        delta_gc_true = pd.read_csv(os.path.join(input_data_folder, 'delta_true.txt'), header=None,
                                      index_col=None, sep='\t').values
        np.save(npy_name, delta_gc_true)
    print("Loaded data")
    num_dims = delta_gc_true.shape[0]
    
    # true_dists_test = distance.pdist(delta_gc_true.T, metric='sqeuclidean')/num_dims
    true_dists = distance.cdist(delta_gc_true[:, cell_inds_selected_gt].T,
                                delta_gc_true.T, metric="sqeuclidean",) / num_dims
    true_dists_lst.append(true_dists)

In [ ]:
# Calculate distances on tree
import json
cell_ids_lst = []
bonsai_dists_lst = []
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    print(ind_dataset)
    tree_results = os.path.join(bonsai_results, find_latest_tree_folder(bonsai_results))
    with open(os.path.join(tree_results, 'metadata.json'), 'r') as file:
        metadata_json = json.load(file)
    cell_ids = metadata_json['cellIds']
    cell_ids_lst.append(cell_ids)
    bonsai_dists = get_pdists_from_subset(os.path.join(tree_results, 'tree.nwk'), 
                                          cell_ids, cell_ids_selected_lst[ind_dataset])
    bonsai_dists_lst.append(bonsai_dists)

In [ ]:
# Calculate distances on iterative tree
import json
cell_ids_lst = []
bonsai_iterative_dists_lst = []
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders_iterative):
    print(ind_dataset)
    tree_results = os.path.join(bonsai_results, find_latest_tree_folder(bonsai_results))
    with open(os.path.join(tree_results, 'metadata.json'), 'r') as file:
        metadata_json = json.load(file)
    cell_ids = metadata_json['cellIds']
    cell_ids_lst.append(cell_ids)
    bonsai_dists = get_pdists_from_subset(os.path.join(tree_results, 'tree.nwk'), 
                                          cell_ids, cell_ids_selected_lst[ind_dataset])
    bonsai_iterative_dists_lst.append(bonsai_dists)

In [ ]:
true_dists_lst[0].shape

In [ ]:
fig, ax = plt.subplots()
ax.scatter(true_dists_lst[0], bonsai_iterative_dists_lst[0], s=2)

In [ ]:
# if 'pca' in methods:
#     pca_dists_lst = []
#     for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
#         print(ind_dataset)
#         pca_projected = pca_projected_lst[ind_dataset]
#         for n_comps, pca_proj in pca_projected.items():
#             if n_comps != 2:
#                 continue
#             pca_dists = distance.pdist(pca_proj.T, metric='sqeuclidean') / 2
#             pca_dists_lst.append(pca_dists)

In [ ]:
# if 'umap' in methods:
#     umap_dists_lst = []
#     for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
#         print(ind_dataset)
#         umap_projected = umap_projected_lst[ind_dataset]
#         umap_proj = umap_projected[N_PCS_UMAP]
#         umap_dists = distance.pdist(umap_proj.T, metric='sqeuclidean') / 2
#         umap_dists_lst.append(umap_dists)

In [ ]:
# if 'phate' in methods:
#     phate_dists_lst = []
#     for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
#         print(ind_dataset)
#         phate_projected = phate_projected_lst[ind_dataset]
#         phate_proj = phate_projected[N_PCS_PHATE]
#         phate_dists = distance.pdist(phate_proj.T, metric='sqeuclidean') / 2
#         phate_dists_lst.append(phate_dists)

In [ ]:
# if 'DTNE' in methods:
#     dtne_dists_lst = []
#     for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
#         print(ind_dataset)
#         dtne_projected = dtne_projected_lst[ind_dataset]
#         dtne_proj = dtne_projected[N_PCS_DTNE]
#         dtne_dists = distance.pdist(dtne_proj.T, metric='sqeuclidean') / 2
#         dtne_dists_lst.append(dtne_dists)

In [ ]:
# Initialize some list for storing information about the different tools
tool_objcts_lst = []

## Plot everything in big figure without the box-plots

In [ ]:
figs_lst = []
axs_lst = []
# ncols = 5 if INCLUDE_SANITY else 4
ncols = 2
for ind_dataset, dataset in enumerate(dataset_descr_lst):
    fig, axs = plt.subplots(nrows=1, ncols=ncols, figsize=(2*3, 6))
    axs = axs[None, :]
    # Make the 2nd row share y-axis with the first subplot in the 2nd row
    for i in range(1, ncols):  # columns 1 to 3
        axs[0, i].sharey(axs[0, 0])
    figs_lst.append(fig)
    axs_lst.append(axs)
    for ax in axs.flatten():
        ax.set_box_aspect(1)
    # ax.axis('off')
    plt.tight_layout()
    plt.subplots_adjust(left=0, right=1.0, bottom=0.12, top=0.88)
#     plt.tight_layout()
    fig.suptitle(dataset_descr_lst[ind_dataset], fontsize=20)

In [ ]:
tool_dists

In [ ]:
true_dists.shape, tool_dists_ds.shape

In [ ]:
tools_lst = ['bonsai_normal', 'bonsai_iterative']
toolname_map = {'bonsai_normal': 'Standard Bonsai', 'bonsai_iterative': "Iterative Bonsai"}
fig, axs = plt.subplots(nrows=2, ncols=len(bonsai_dists_lst), figsize=(12,6))
# bonsai_results = bonsai_results_folders[0]
dists_lst = {}
dists_lst['bonsai_normal'] = bonsai_dists_lst
dists_lst['bonsai_iterative'] = bonsai_iterative_dists_lst

for ind_dataset, bonsai_dists in enumerate(bonsai_dists_lst):
    true_dists = true_dists_lst[ind_dataset]
    for ind_tool, tool in enumerate(tools_lst):
        tool_dists_ds = dists_lst[tool][ind_dataset]
        ax = axs[ind_tool, ind_dataset]
        ax.scatter(true_dists, tool_dists_ds, s=2)
        ax.axline((0,0), slope=1, linestyle='--', c='black')
        ax.set_xlabel('True distances')
        ax.set_ylabel('{} distances'.format(toolname_map[tool]))

plt.tight_layout()

In [ ]:
fig.savefig('figures/pairwisedists_standarditerative.png', dpi=300)
# fig.savefig('figures/pairwisedists_standarditerative.svg')

In [ ]:
def compare_pdists_to_truth_per_cell_here(datasets, make_fig=True, axs=None, title='', YLABEL=True, XLABEL=True,
                                     first_title=None, set_lims=True, corr_measure='Rsq', return_Rvals=False,
                                     flip_axes=False, density=False, bins=20, share_y=False, loglog_corr=False,
                                    dists_are_flat=True):
    if make_fig and (axs is None):
        n_datasets = len(datasets) - 1  # Minus 1 to subtract for true dataset
        n_rows = int(np.ceil(np.sqrt(n_datasets)))
        n_cols = int(np.ceil(n_datasets / n_rows))
        fig, axs = plt.subplots(n_rows=n_rows, n_cols=n_cols, figsize=(8, 6))

    true_dataset_ind = [ind for ind, dataset in enumerate(datasets) if dataset.data_type == 'delta_true']
    if len(true_dataset_ind) != 1:
        print("Exactly one of the provided dataset should contain the true nearest neighbours")
        exit()
    else:
        true_dataset_ind = true_dataset_ind[0]
        true_dataset = datasets[true_dataset_ind]
        pdists_true = true_dataset.distances

    # n_cells = nn_mat_true.shape[0] + 1
    dataset_counter = -1
    avg_rel_diffs = [None]
    R_vals = {}
    for ind_dataset, dataset in enumerate(datasets):
        if ind_dataset == true_dataset_ind:
            continue
        dataset_counter += 1
        try:
            ax = axs.flatten()[dataset_counter]
            fontsize = 'x-small' if (len(axs.flatten()) > 4) else 'large'
        except AttributeError:
            ax = axs
            fontsize = 'x-small'
        fontsize = 'large'
        print("Comparing pairwise distances of dataset: '" + dataset.data_type + "'.")
        if dists_are_flat:
            pdists_data = squareform(dataset.distances)
            true_dists = squareform(pdists_true)
        else:
            pdists_data = dataset.distances
            true_dists = pdists_true
        # log_truedists = squareform(np.log(pdists_true))

        n_cells = true_dists.shape[0]
        if dists_are_flat:
            # Remove the diagonal element, which is uninformative for distances
            mask = ~np.eye(n_cells, dtype=bool)
            true_dists = true_dists[mask].reshape(n_cells, n_cells-1)
            pdists_data = pdists_data[mask].reshape(n_cells, n_cells-1)

        # if loglog_corr:
        #     log_truedists = np.log(true_dists)
        # else:
        #     log_truedists = true_dists

        recalc_pearsonRs = False
        if dataset.pearsonRs is None:
            dataset.pearsonRs = np.zeros(n_cells)
            dataset.avg_rel_errors = np.zeros(n_cells)
            recalc_pearsonRs = True
        if make_fig:
            if recalc_pearsonRs:
                # Get correlation between distances:
                for cell_ind in range(n_cells):
                    # THIS_FIG = (cell_ind % int(np.ceil(n_cells / 10)) == 0)
                    THIS_FIG = False
                    pdists_data_cell = pdists_data[cell_ind, :]
                    # log_truedists_cell = log_truedists[cell_ind, :]
                    # pdists_true_cell = squareform(pdists_true)[cell_ind, :]
                    pdists_true_cell = true_dists[cell_ind, :]

                    if THIS_FIG:
                        fig_cell, ax_cell = plt.subplots()
                        if not dataset.data_type == 'delta_true':
                            if flip_axes:
                                ax_cell.scatter(pdists_data_cell, pdists_true[cell_ind, :], s=2.5,
                                                color=dataset.data_type_color, zorder=0, alpha=.5)
                            else:
                                ax_cell.scatter(pdists_true_cell, pdists_data_cell, s=2.5,
                                                color=dataset.data_type_color, zorder=0, alpha=.5)

                    nonzeros_true_cell = pdists_true_cell != 0
                    nonzeros_cell = pdists_data_cell != 0
                    nonzeros_all = nonzeros_cell * nonzeros_true_cell
                    # log_truedists_nonzeros_cell = log_truedists_cell[nonzeros_all]
                    if loglog_corr:
                        truedists_transformed_cell = np.log(pdists_true_cell[nonzeros_all])
                        datadists_transformed_cell = np.log(pdists_data_cell[nonzeros_all])
                    else:
                        truedists_transformed_cell = pdists_true_cell
                        datadists_transformed_cell = pdists_data_cell

                    # # TODO: Maybe remove later
                    # lt1 = log_truedists_nonzeros > -1e9
                    # log_truedists_lt1 = log_truedists_nonzeros[lt1]
                    # log_datadists_lt1 = log_datadists[lt1]

                    # Clog = np.cov(np.vstack((log_truedists_nonzeros - np.mean(log_truedists_nonzeros), log_datadists - np.mean(log_datadists))))
                    Clog = np.cov(np.vstack(
                        (truedists_transformed_cell - np.mean(truedists_transformed_cell),
                         datadists_transformed_cell - np.mean(datadists_transformed_cell))))
                    pearsonR = Clog[0, 1] / np.sqrt(Clog[0, 0] * Clog[1, 1])
                    dataset.pearsonRs[cell_ind] = pearsonR
                    dataset.avg_rel_errors[cell_ind] = np.mean(np.abs(
                        (pdists_true_cell[nonzeros_all] - pdists_data_cell[nonzeros_all]) / pdists_true_cell[nonzeros_all]))
                    # dataset.PearsonR = corrlog
                    if THIS_FIG:
                        eigVals, eigVecs = np.linalg.eig(Clog)
                        max_eigval = np.argmax(eigVals)
                        slopepca1log = eigVecs[1, max_eigval] / eigVecs[0, max_eigval]
                        if flip_axes:
                            Clog_flipped = np.cov(np.vstack(
                                (datadists_transformed_cell - np.mean(datadists_transformed_cell),
                                 truedists_transformed_cell - np.mean(truedists_transformed_cell))))
                            eigVals_flipped, eigVecs_flipped = np.linalg.eig(Clog_flipped)
                            max_eigval_flipped = np.argmax(eigVals_flipped)
                            slopepca1log_flipped = eigVecs_flipped[1, max_eigval_flipped] / eigVecs_flipped[
                                0, max_eigval_flipped]
                        # regLineXlog = np.linspace(log_truedists_nonzeros.min(), log_truedists_nonzeros.max(), 20)
                        regLineXlog = np.linspace(truedists_transformed_cell.min(), truedists_transformed_cell.max(), 20)
                        # regLineYlog = slopepca1log * (regLineXlog - log_truedists_nonzeros.mean()) + log_datadists.mean()
                        regLineYlog = slopepca1log * (
                                regLineXlog - truedists_transformed_cell.mean()) + datadists_transformed_cell.mean()

                    # linregress_res = linregress(x=log_truedists_nonzeros, y=log_datadists)
                    # TODO: Add back later
                    # linregress_res_Y_log = regLineXlog * linregress_res.slope + linregress_res.intercept

                    # TODO: Remove later
                    # from sklearn.linear_model import LinearRegression
                    # x = np.array(log_truedists).reshape((-1, 1))
                    # y = np.array(log_datadists)
                    # model = LinearRegression(fit_intercept=False)
                    # model.fit(x, y)
                    # r_sq = model.score(x, y)
                    # intercept = model.intercept_
                    # slope = model.coef_
                    # regLineYlog = intercept + slope * regLineXlog
                    if THIS_FIG:
                        if not flip_axes:
                            if loglog_corr:
                                ax_cell.plot(np.exp(regLineXlog), np.exp(regLineYlog), '--', lw=2, c='black', zorder=10)
                            else:
                                ax_cell.plot(regLineXlog, regLineYlog, '--', lw=2, c='black', zorder=10)
                        else:
                            if loglog_corr:
                                ax_cell.plot(np.exp(regLineYlog), np.exp(regLineXlog), '--', lw=2, c='black', zorder=10)
                            else:
                                ax_cell.plot(regLineYlog, regLineXlog, '--', lw=2, c='black', zorder=10)
                        # ax.plot(np.exp(regLineXlog), np.exp(linregress_res_Y_log), '--', lw=2, c='black', zorder=12)
                        # ax.text(0.98, 0.1, "Slope: {:.2f},\nR={:.2f}".format(linregress_res.slope, linregress_res.rvalue),
                        #         horizontalalignment='right', verticalalignment='bottom', transform=ax.transAxes, fontsize=fontsize)
                        if corr_measure == 'Rsq':
                            corr_label = pearsonR ** 2
                        elif corr_measure == '-log(1-Rsq)':
                            corr_label = -np.log10(1 - pearsonR ** 2)
                        if not flip_axes:
                            ax_cell.text(0.98, 0.01,
                                         "Slope: {:.2f},\n{}={:.2f}".format(slopepca1log, corr_measure, corr_label),
                                         horizontalalignment='right', verticalalignment='bottom',
                                         transform=ax_cell.transAxes,
                                         fontsize=fontsize)
                        else:
                            ax_cell.text(0.98, 0.01,
                                         "Slope: {:.2f},\n{}={:.2f}".format(slopepca1log_flipped, corr_measure, corr_label),
                                         horizontalalignment='right', verticalalignment='bottom',
                                         transform=ax_cell.transAxes,
                                         fontsize=fontsize)
                    # ax.text(0.98, 0.01, "-log(1-R^2)={:.2f}".format(-np.log(1 - corrlog ** 2)),
                    #         horizontalalignment='right', verticalalignment='bottom', transform=ax.transAxes, fontsize=fontsize)

                    # ax.text("Correlation of %f. Fitted line: y = %f + %f x\" %(corr, regLineY[0], slopepca1))
                    # box = ax.get_position()
                    # ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
                    # ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
                    if set_lims and THIS_FIG:
                        if not flip_axes:
                            ax_cell.set_xlim(np.exp(truedists_transformed_cell.min()),
                                             np.exp(truedists_transformed_cell.max()))
                            # ax.set_ylim(min(np.exp(log_truedists_nonzeros.min()), np.exp(regLineYlog.min())), max(np.exp(log_truedists_nonzeros.max()), np.exp(regLineYlog.max())))
                            ax_cell.set_ylim(np.exp(truedists_transformed_cell.min()),
                                             np.exp(truedists_transformed_cell.max()))
                        else:
                            ax_cell.set_ylim(np.exp(truedists_transformed_cell.min()),
                                             np.exp(truedists_transformed_cell.max()))
                            # ax.set_ylim(min(np.exp(log_truedists_nonzeros.min()), np.exp(regLineYlog.min())), max(np.exp(log_truedists_nonzeros.max()), np.exp(regLineYlog.max())))
                            ax_cell.set_xlim(np.exp(truedists_transformed_cell.min()),
                                             np.exp(truedists_transformed_cell.max()))
                    if THIS_FIG and loglog_corr:
                        ax_cell.set_xscale('log')
                        ax_cell.set_yscale('log')

            ax.hist(dataset.pearsonRs, bins=bins, color=dataset.data_type_color, range=(-0.1, 1), density=density)
            # if dataset_counter == 0:
            if (dataset_counter == len(datasets) - 2) and XLABEL:
                ax.set_xlabel('R-squared values between\ninferred and true distances per cell', fontsize=fontsize)
            fam = dataset.data_type.split('_')[0]
            if YLABEL:
                ax.set_ylabel("Numbers of cells")
                # if fam != 'bonsai':
                #     ax.set_ylabel('{}: Inferred \nsquared distances'.format(fam), fontsize=fontsize)
                # else:
                #     ax.set_ylabel('Bonsai: Summed \nbranch lengths', fontsize=fontsize)
            if first_title is not None:
                if ind_dataset == 0:
                    ax.set_title(first_title)
            else:
                ax.set_title(dataset.data_id)
    if return_Rvals:
        return avg_rel_diffs, R_vals
    else:
        return avg_rel_diffs

In [ ]:
# Create histogram of correlations figures
RECALCULATE = True
RECALCULATE = RECALCULATE or not len(tool_objcts_lst)
if RECALCULATE:
    tool_objcts_lst = []
    
# if INCLUDE_SANITY:
#     tools_lst.append('sanity')
for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    print("\n\nTreating dataset {}\n".format(dataset_descr_lst[ind_dataset]))
    true_dists = true_dists_lst[ind_dataset]
    tools_dists_dict = {}
    if 'bonsai_normal' in tools_lst:
        tools_dists_dict['bonsai_normal'] = bonsai_dists_lst[ind_dataset]
    if 'bonsai_iterative' in tools_lst:
        tools_dists_dict['bonsai_iterative'] = bonsai_iterative_dists_lst[ind_dataset]
    if RECALCULATE:
        tool_objcts = {}
        true_objct = Dataset(distances=true_dists_lst[ind_dataset], data_type='delta_true', data_id='delta_true', color_types=tools_lst)
        true_objct.true_dataset_ranks = None
        for ind_tool, tool in enumerate(tools_lst):
            data_id = tool + dataset_descr_lst[ind_dataset]
            tool_objcts[tool] = Dataset(distances=tools_dists_dict[tool], data_type=tool, data_id=data_id, color_types=tools_lst)
    else:
        tool_objcts = tool_objcts_lst[ind_dataset]
    for ind_tool, tool in enumerate(tools_lst):
        tool_objct = tool_objcts[tool]
        # if ind_tool != 0:
        #     continue
#         fig, ax = plt.subplots(figsize=(3,3))
        fig = figs_lst[ind_dataset]
        ax = axs_lst[ind_dataset][0, ind_tool]
        ax.set_box_aspect(1)
        # n_neighbours_list = compare_nearest_neighbours_to_truth([true_objct, tool_objct], make_fig=False, max_neighbours=600, ax=None,
        #                                          only_powers_of_2=True,
        #                                          title='')
        avg_rel_diffs, R_vals = compare_pdists_to_truth_per_cell_here([true_objct, tool_objct], make_fig=True, axs=ax, set_lims=False, 
                                                                      return_Rvals=True, XLABEL=False, YLABEL=False, flip_axes=True, 
                                                                      first_title=' ', loglog_corr=False, dists_are_flat=False, bins=100)
    if RECALCULATE:
        tool_objcts_lst.append(tool_objcts)


In [ ]:
tool_name_map = {'bonsai_normal': "Default Bonsai", 'bonsai_iterative': "Backbone-based Bonsai"}
# Create boxplots in separate figure as well, now grouped by dataset
boxprops = dict(linewidth=0.05)
flierprops = dict(markersize=2, markeredgewidth=0.5)
medianprops = dict(color='black', linewidth=1)
nrows = 1
ncols = int(np.ceil(len(dataset_ids)/nrows))
fig_bp, axs_bp = plt.subplots(figsize=(3*ncols,5*nrows), nrows=nrows, ncols=ncols)

axs_bp = np.array([axs_bp])
axs_bp = axs_bp.flatten()
for ax in axs_bp:
    ax.axis('off')

for ind_dataset, bonsai_results in enumerate(bonsai_results_folders):
    ax = axs_bp[ind_dataset]
    ax.axis('on')
    dataset_descr = dataset_descr_lst[ind_dataset]
    pearsonRs = []
    tool_names = []
    for ind_tool, tool in enumerate(tools_lst):
        # if (not INCLUDE_SANITY) and (tool == 'sanity'):
        #     continue
        tool_objct = tool_objcts_lst[ind_dataset][tool]
        tool_names.append(tool_name_map[tool])
        pearsonRs.append(tool_objct.pearsonRs)

    spacing = 0.3  # < 1 → closer together
    positions = np.arange(len(pearsonRs)) * spacing + 1
    
    ax = axs_bp[ind_dataset]
    bplot = ax.boxplot(pearsonRs, whis=(5, 95), positions=positions, labels=tool_names, patch_artist=True, 
                           flierprops=flierprops, medianprops=medianprops, boxprops=boxprops)
    # fill with colors
    for ind_patch, patch in enumerate(bplot['boxes']):
        patch.set_facecolor(color=tool_objcts_lst[ind_dataset][tools_lst[ind_patch]].data_type_color)
    # ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_xticks(positions)
    ax.set_xticklabels(tool_names, rotation=45, ha="right")
    ax.set_xlim(positions[0] - 0.2, positions[-1] + 0.2)
    ax.set_ylim(-0.05,1.05)
    ax.set_title(dataset_descr)
    if ind_dataset == 0:
        ax.set_ylabel('Correlation (Pearson R) between\ntrue and visualized distances')
plt.tight_layout()

In [ ]:
# Store boxplots in a figure.
fig_bp.savefig('figures/boxplots_distscomp_standarditerative.png', dpi=300)
fig_bp.savefig('figures/boxplots_distscomp_standarditerative.svg')

In [ ]:
# Change some titles and labels
# Create boxplots in separate figure as well, now grouped by dataset

for ind_fig, fig_obj in enumerate(figs_lst):
    axs = axs_lst[ind_fig]
    if 'bonsai_normal' in tools_lst:
        axs[0, tools_lst.index('bonsai_normal')].set_title('Bonsai standard', fontsize=16)
    if 'bonsai_iterative' in tools_lst:
        axs[0, tools_lst.index('bonsai_iterative')].set_title('Bonsai iterative', fontsize=16)
    for ind_ax, ax in enumerate(axs[0,:]):
        ax.set_xlabel("Pearson R", fontsize=12)

    # axs[0,0].set_ylabel('Visualization', fontsize=16)
    axs[0,0].set_ylabel('Number of cells', fontsize=12)
#     axs[1,1].set_ylabel('True squared\ndistances')
#     axs[1,1].set_xlabel('Inferred squared\ndistances')
#     axs[2,1].set_ylabel('R-squared')
    # axs[1,0].axis('off')
    # axs[1,0].text(0.5, 0.5, "Pearson R-values\nfor correlation of\ninferred and true\ndistances between\neach cell and all\nothers.",
    #                 horizontalalignment='center', verticalalignment='center', transform=axs[1,0].transAxes, fontsize=16)
    fig_obj.subplots_adjust(bottom=0.05, top=0.88, left=0.1, right=.9)
    # fig_obj.text(0.03, 0.9, 'A)', fontsize=14, fontweight='bold', va='top', ha='left')
    # fig_obj.text(0.03, 0.45, 'B)', fontsize=14, fontweight='bold', va='top', ha='left')


In [ ]:
# Save figures
figure_folder = 'figures'
Path(figure_folder).mkdir(parents=True, exist_ok=True)
for ind_fig, fig_obj in enumerate(figs_lst):
    dataset_descr_label = dataset_descr_lst[ind_fig].replace(' ','_')
    fig_obj.savefig(os.path.join(figure_folder, '{}_overview_figure_wo_boxplots.png'.format(dataset_descr_label)), dpi=300)
    fig_obj.savefig(os.path.join(figure_folder, '{}_overview_figure_wo_boxplots.svg'.format(dataset_descr_label)))
    print(os.path.join(figure_folder, '{}_overview_figure_wo_boxplots.svg'.format(dataset_descr_label)))

### Show figures

In [ ]:
ind=0
figs_lst[ind]

In [ ]:
ind=1
figs_lst[ind]

In [ ]:
ind=2
figs_lst[ind]

In [ ]:
ind=3
figs_lst[ind]